# On-policy distillation (OPD) through a tiny network, by hand

```
X (2 contexts) -> Z1 = X @ W1 -> H = tanh(Z1) -> logits = H @ W2 -> p = softmax(logits)
each context writes a token y; the teacher grades it: A = log p_teacher[y] - log p_student[y]
loss = mean(-A * log p_student[y])        (A is frozen)
```

The gradient on the logits is the SFT one, `(p - y_onehot) / b`, multiplied by each
row's A. A > 0 pulls the written token up, A < 0 pushes it down.

In [1]:
import numpy as np
import torch
np.set_printoptions(precision=3, suppress=True)

TOKENS = ["A", "B", "C"]
X = np.array([[1.0, 0.0],              # context 1
              [0.0, 1.0]])             # context 2
P_T = np.array([[0.6, 0.3, 0.1],       # teacher for context 1
                [0.1, 0.2, 0.7]])      # teacher for context 2


class Network:
    def __init__(self, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 1, (2, 4))   # input -> hidden. seed 0: [[ 0.126, -0.132, 0.640, 0.105],
                                             #                           [-0.536,  0.362, 1.304, 0.947]]
        self.W2 = rng.normal(0, 1, (4, 3))   # hidden -> scores. seed 0: [[-0.704, -1.265, -0.623],
                                             #                            [ 0.041, -2.325, -0.219],
                                             #                            [-1.246, -0.732, -0.544],
                                             #                            [-0.316,  0.412,  1.043]]

    def probs(self, X):
        z = np.tanh(X @ self.W1) @ self.W2
        e = np.exp(z - z.max(axis=1, keepdims=True))
        return e / e.sum(axis=1, keepdims=True)

    def forward(self, X, y, P_T):
        # 1. Z1     = X @ W1
        # 2. H      = tanh(Z1)
        # 3. logits = H @ W2
        # 4. p      = softmax(logits)
        # 5. logp_y = log p[row, y]               log-prob of the token each row wrote
        # 6. A      = log P_T[row, y] - logp_y    teacher's grade, one per row (frozen)
        # 7. loss   = mean(-A * logp_y)
        # Values in the comments: the default greedy step below (seed 0, X = identity,
        # y = [1, 2] = [B, C]). Rows are context 1 and context 2.
        acts = {}
        b = X.shape[0]                                             # 2
        acts["X"], acts["y"], acts["b"] = X, y, b
        acts["Z1"] = X @ self.W1                                   # 1: [[ 0.126, -0.132, 0.640, 0.105],
                                                                   #     [-0.536,  0.362, 1.304, 0.947]]  (= W1, since X is identity)
        acts["H"] = np.tanh(acts["Z1"])                            # 2: [[ 0.125, -0.131, 0.565, 0.105],
                                                                   #     [-0.490,  0.347, 0.863, 0.738]]
        acts["logits"] = acts["H"] @ self.W2                       # 3: [[-0.831, -0.224, -0.248],
                                                                   #     [-0.950, -0.514,  0.530]]
        e = np.exp(acts["logits"] - acts["logits"].max(axis=1, keepdims=True))
        acts["p"] = e / e.sum(axis=1, keepdims=True)               # 4: [[0.216, 0.397, 0.387],
                                                                   #     [0.144, 0.223, 0.633]]
        acts["logp_y"] = np.log(acts["p"][np.arange(b), y])        # 5: [log 0.397, log 0.633] = [-0.925, -0.457]
        acts["A"] = np.log(P_T[np.arange(b), y]) - acts["logp_y"]  # 6: [log 0.3 + 0.925, log 0.7 + 0.457] = [-0.279, +0.101]
        self.acts = acts
        return np.mean(-acts["A"] * acts["logp_y"])                # 7: mean(-[-0.279, 0.101] * [-0.925, -0.457]) = -0.106

    def backward(self, verbose=False):
        acts = self.acts
        b, p, y, A = acts["b"], acts["p"], acts["y"], acts["A"]

        # 7 -> 4: loss -> logits. SFT gradient (p - y_onehot) / b, times each row's A.
        # A is a frozen number per row, so it only scales that row.
        y_onehot = np.zeros_like(p)
        y_onehot[np.arange(b), y] = 1                  # [[0, 1, 0],          context 1 wrote B
                                                       #  [0, 0, 1]]          context 2 wrote C
        # p - y_onehot                                 = [[ 0.216, -0.603,  0.387],
        #                                                 [ 0.144,  0.223, -0.367]]
        # A[:, None]                                   = [[-0.279],    one A per row, as a column
        #                                                 [ 0.101]]    so it scales the whole row
        g_logits = A[:, None] * (p - y_onehot) / b     # [[-0.030,  0.084, -0.054],   row 1: A < 0 reverses SFT's sign on B (-0.603 -> +0.084): B pushed down
                                                       #  [ 0.007,  0.011, -0.018]]   row 2: A > 0 keeps SFT's sign on C (-0.367 -> -0.018): C pulled up

        # 3: logits = H @ W2  ->  dW2 = H.T @ g_logits, and send the gradient back to H.
        dW2 = acts["H"].T @ g_logits                   # [[-0.007,  0.005,  0.002],
                                                       #  [ 0.006, -0.007,  0.001],
                                                       #  [-0.011,  0.057, -0.047],
                                                       #  [ 0.002,  0.017, -0.019]]
        g_H = g_logits @ self.W2.T                     # [[-0.052, -0.185,  0.005, -0.012],
                                                       #  [-0.008, -0.022, -0.007, -0.017]]

        # 2: H = tanh(Z1). Slope of tanh is 1 - tanh^2.
        #    1 - H**2                                  = [[0.984, 0.983, 0.681, 0.989],
        #                                                 [0.760, 0.880, 0.256, 0.455]]
        g_Z1 = g_H * (1 - acts["H"] ** 2)              # [[-0.051, -0.182,  0.004, -0.012],
                                                       #  [-0.006, -0.019, -0.002, -0.008]]

        # 1: Z1 = X @ W1  ->  dW1 = X.T @ g_Z1
        dW1 = acts["X"].T @ g_Z1                       # same as g_Z1, since X is identity

        if verbose:
            print("A per row    :", A)
            print("p - y_onehot :\n", p - y_onehot)
            print("g_logits     :\n", g_logits)
            print("dW2          :\n", dW2)
            print("dW1          :\n", dW1)
        return {"dW1": dW1, "dW2": dW2}

## One greedy step

Each context writes its top token. Context 1 writes B, which the student likes more
than the teacher does (A < 0, push down). Context 2 writes C, which the teacher likes
more (A > 0, pull up).

In [2]:
net = Network()
p0 = net.probs(X)
y = p0.argmax(axis=1)                 # greedy
loss = net.forward(X, y, P_T)
print("student p:\n", p0)
print("written  :", [TOKENS[i] for i in y], "  loss:", round(loss, 4))
grads = net.backward(verbose=True)

student p:
 [[0.216 0.397 0.387]
 [0.144 0.223 0.633]]
written  : ['B', 'C']   loss: -0.1061
A per row    : [-0.279  0.101]
p - y_onehot :
 [[ 0.216 -0.603  0.387]
 [ 0.144  0.223 -0.367]]
g_logits     :
 [[-0.03   0.084 -0.054]
 [ 0.007  0.011 -0.018]]
dW2          :
 [[-0.007  0.005  0.002]
 [ 0.006 -0.007  0.001]
 [-0.011  0.057 -0.047]
 [ 0.002  0.017 -0.019]]
dW1          :
 [[-0.051 -0.182  0.004 -0.012]
 [-0.006 -0.019 -0.002 -0.008]]


### The sign rule

In `p - y_onehot`, the **written** token's entry is `p - 1`, always negative; every
other token's entry is its `p`, positive. Multiplying by A:

| | written token | other tokens | effect (training subtracts the gradient) |
|---|---|---|---|
| A > 0 (context 2, C) | 0.633 - 1 = -0.367 -> stays negative (-0.018) | positive | written token goes **up** |
| A < 0 (context 1, B) | 0.397 - 1 = -0.603 -> flips to positive (+0.084) | negative | written token goes **down** |

**A negative gradient on the written token's logit means "raise this logit".** The
logit can only move through the weights, so the gradient flows back:

- `dW2 = H.T @ g_logits`: a weight into that token gets gradient (hidden activation) x
  (token's gradient). With the token's gradient negative, weights from active units
  (H > 0) get a negative gradient and **increase**; weights from units with H < 0
  **decrease**. Both raise the logit. Context 2, unit 2: 0.863 x (-0.018) = -0.016, so
  its weight into C goes up.
- `g_H`, then `dW1`: the same gradient continues into W1, shifting the hidden units in
  the direction that favors the token.

## Check against PyTorch

In [3]:
W1 = torch.tensor(net.W1, requires_grad=True)
W2 = torch.tensor(net.W2, requires_grad=True)
logp = torch.log_softmax(torch.tanh(torch.tensor(X) @ W1) @ W2, dim=1)
rows, cols = torch.arange(2), torch.tensor(y)
logp_y = logp[rows, cols]
A = (torch.log(torch.tensor(P_T))[rows, cols] - logp_y).detach()   # stop-gradient
(-A * logp_y).mean().backward()
print("dW1 matches:", np.allclose(grads["dW1"], W1.grad.numpy()))
print("dW2 matches:", np.allclose(grads["dW2"], W2.grad.numpy()))

dW1 matches: True
dW2 matches: True


## What a gradient is

For each number: **if I nudge it up a little, how much does the loss change?**
Check it on context 2's logits (it wrote C, A = +0.101): nudge each by 0.01.

In [4]:
a = net.acts
z2 = a["logits"][1].copy()                     # [-0.950, -0.514, 0.530]
A2 = a["A"][1]                                 # +0.101, frozen

def row2_loss(z):
    p = np.exp(z) / np.exp(z).sum()
    return -A2 * np.log(p[2]) / 2              # row 2's share of the mean loss (b = 2)

L0 = row2_loss(z2)
for k in range(3):
    z = z2.copy()
    z[k] += 0.01
    print(f"nudge {TOKENS[k]}'s logit +0.01: loss change {row2_loss(z) - L0:+.6f}  ->  gradient {(row2_loss(z) - L0) / 0.01:+.3f}")

y_onehot = np.eye(3)[a["y"][1]]
print("g_logits row 2 = A * (p - y_onehot) / b:", A2 * (a["p"][1] - y_onehot) / 2)   # [0.007, 0.011, -0.018]

nudge A's logit +0.01: loss change +0.000073  ->  gradient +0.007
nudge B's logit +0.01: loss change +0.000113  ->  gradient +0.011
nudge C's logit +0.01: loss change -0.000184  ->  gradient -0.018
g_logits row 2 = A * (p - y_onehot) / b: [ 0.007  0.011 -0.018]


Same numbers. Training then steps **against** the gradient,
`new = old - lr * gradient`, because the gradient points to where the loss goes **up**:

| | A | B | C |
|---|---|---|---|
| logit before | -0.950 | -0.514 | 0.530 |
| gradient | +0.007 | +0.011 | -0.018 |
| logit after (lr = 1) | -0.957 (down) | -0.525 (down) | 0.548 (up) |

## Update the weights: `W = W - lr * dW`

In [5]:
net.W1 -= 1.0 * grads["dW1"]
net.W2 -= 1.0 * grads["dW2"]
p1 = net.probs(X)
for i in range(2):
    w = TOKENS[y[i]]
    print(f"context {i+1}: wrote {w}   p({w}) {p0[i, y[i]]:.3f} -> {p1[i, y[i]]:.3f}   teacher {P_T[i, y[i]]:.1f}")

context 1: wrote B   p(B) 0.397 -> 0.288   teacher 0.3
context 2: wrote C   p(C) 0.633 -> 0.659   teacher 0.7


## Train (sampling)

Each step, each context samples 32 tokens from the student and we step on the
average gradient. Sampling, not greedy, so every token gets written and corrected.

In [6]:
rng = np.random.default_rng(1)
net = Network()
for step in range(301):
    P = net.probs(X)
    Xb = np.repeat(X, 32, axis=0)
    Tb = np.repeat(P_T, 32, axis=0)
    yb = np.concatenate([rng.choice(3, 32, p=P[i]) for i in range(2)])
    net.forward(Xb, yb, Tb)
    g = net.backward()
    net.W1 -= 0.5 * g["dW1"]
    net.W2 -= 0.5 * g["dW2"]
    if step in (0, 100, 300):
        P = net.probs(X)
        print(f"step {step:3d}  context 1 {P[0]}  context 2 {P[1]}")
print("teacher   context 1", P_T[0], "  context 2", P_T[1])

step   0  context 1 [0.243 0.393 0.363]  context 2 [0.146 0.219 0.634]
step 100  context 1 [0.6 0.3 0.1]  context 2 [0.1 0.2 0.7]
step 300  context 1 [0.6 0.3 0.1]  context 2 [0.1 0.2 0.7]
teacher   context 1 [0.6 0.3 0.1]   context 2 [0.1 0.2 0.7]
